# 070 CASE 010 — Getting Started

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/imintengine/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_010-Getting-Started.ipynb)

**Purpose:** Template and onboarding notebook for the SDL 3.0 use-case library. Every case notebook follows this 6-cell structure.

**Partners:** RISE · AI Sweden · Luleå tekniska universitet · Rymdstyrelsen

**Data sources:** ESA Copernicus Sentinel-2 L2A via [Digital Earth Sweden](https://digitalearth.se), CDSE fallback, or synthetic data for offline use.

**License:** CC0 1.0 Universal (notebook content). ImintEngine analyzers retain original licenses — see [THIRD_PARTY_LICENSES.md](https://github.com/TobiasEdman/imintengine/blob/main/THIRD_PARTY_LICENSES.md).

## 1. Method

Runs a minimal ImintEngine analysis over a small AOI using the `spectral` analyzer (NDVI/NDWI/NDBI/EVI/NBR).

**Execution flow:**

```
AOI + date → LocalExecutor.build_job() → fetch Sentinel-2 → run_job() → visualize
                       │
                       └─ synthetic fallback if DES/CDSE unreachable
```

**Why `build_job + run_job` instead of `execute`:** we need access to the built `IMINTJob` (for `job.rgb`) *and* the `IMINTResult` (for analyzer outputs). The one-shot `executor.execute(...)` returns only the `IMINTResult`.

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
from imint.engine import run_job
import matplotlib.pyplot as plt
import folium

# Helper: access analyzer outputs by name (IMINTResult.analyzer_results is a list)
def get_outputs(result, name):
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None

# Small AOI — Gotland coastline sample
AOI = {"west": 18.0, "south": 57.2, "east": 19.0, "north": 57.8}
DATE = "2024-06-15"

print(f"AOI: {AOI}")
print(f"Date: {DATE}")

## 3. Run analysis

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/getting_started",
    config_path="config/analyzers.yaml",
)

# Build the job (fetches data, falls back to synthetic if offline)
job = executor.build_job(date=DATE, coords=AOI)

# Run analyzers
result = run_job(job)

print(f"Job ID: {result.job_id}")
print(f"Success: {result.success}")
successful = [ar.analyzer for ar in result.analyzer_results if ar.success]
failed = [(ar.analyzer, ar.error) for ar in result.analyzer_results if not ar.success]
print(f"Successful analyzers: {successful}")
if failed:
    print("Failed analyzers (expected on incomplete band sets):")
    for name, err in failed:
        print(f"  {name}: {(err or '')[:90]}")

## 4. Visualize

In [ ]:
# Interactive Leaflet map centered on AOI
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")
folium.Rectangle(
    bounds=[[AOI["south"], AOI["west"]], [AOI["north"], AOI["east"]]],
    color="#ff7800", weight=2, fill=True, fill_opacity=0.1,
    popup=f"AOI · {DATE}",
).add_to(m)
m

In [ ]:
# Static plots: RGB composite + NDVI + NDWI
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(job.rgb)
axes[0].set_title("Sentinel-2 RGB")
axes[0].axis("off")

spec = get_outputs(result, "spectral")
if spec is not None:
    indices = spec["indices"]
    axes[1].imshow(indices["NDVI"], cmap="RdYlGn", vmin=-1, vmax=1)
    axes[1].set_title("NDVI (vegetation)")
    axes[1].axis("off")

    axes[2].imshow(indices["NDWI"], cmap="Blues", vmin=-1, vmax=1)
    axes[2].set_title("NDWI (water)")
    axes[2].axis("off")
else:
    for ax in axes[1:]:
        ax.text(0.5, 0.5, "spectral analyzer unavailable", ha="center", va="center")
        ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Interpretation & template usage

**What this demonstrated:**

- End-to-end flow: AOI → data fetch → analyzer run → visualization in < 5 minutes
- Reproducibility: every cell is deterministic given the same AOI/date + data source
- Fallback strategy: synthetic data keeps the notebook runnable without DES credentials

### Verified ImintEngine API (copy this pattern)

```python
from executors.local import LocalExecutor
from imint.engine import run_job

executor = LocalExecutor(output_dir="outputs/my_case")
job = executor.build_job(date="2024-06-15", coords={...})
result = run_job(job)  # IMINTResult

# Access outputs:
for ar in result.analyzer_results:
    if ar.analyzer == "spectral" and ar.success:
        ndvi = ar.outputs["indices"]["NDVI"]
        break

# Access source imagery from the JOB (not the result):
rgb = job.rgb
bands = job.bands  # dict of band-name -> array
```

### Template usage

1. Copy this notebook to `NNN_CASE_NNN-Name.ipynb`
2. Update title/partners/data-sources in cell 1
3. Swap AOI + date for your case
4. Enable case-specific analyzers in `config/analyzers.yaml`
5. Tailor the visualization to your analysis outputs

### Links

- [ImintEngine source](https://github.com/TobiasEdman/imintengine)
- [Digital Earth Sweden](https://digitalearth.se)